In [4]:
import pandas as pd
import numpy as np

print("🚀 SalesNova Feature Engineering")
print("=" * 45)

🚀 SalesNova Feature Engineering


In [5]:
df = pd.read_csv(
    "../data/processed/sales_processed.csv",
    parse_dates=["Date"]
)

print("Dataset shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())

Dataset shape: (844392, 26)
Date range: 2013-01-01 00:00:00 to 2015-07-31 00:00:00


C:\Users\Aaron Kuriyan\AppData\Local\Temp\ipykernel_21668\2342355357.py:1: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [6]:
df = df.sort_values(
    ["Store", "Date"]
).reset_index(drop=True)

print("Data sorted by Store and Date.")

Data sorted by Store and Date.


In [7]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["DayOfWeek"] = df["Date"].dt.dayofweek + 1
df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
df["Quarter"] = df["Date"].dt.quarter

df["IsWeekend"] = (
    df["DayOfWeek"].isin([6, 7])
).astype(int)

print("Time features created.")

Time features created.


In [8]:
df["Sales_Lag_1"] = (
    df.groupby("Store")["Sales"].shift(1)
)

print(df[[
    "Store",
    "Date",
    "Sales",
    "Sales_Lag_1"
]].head(10))

   Store       Date  Sales  Sales_Lag_1
0      1 2013-01-02   5530          NaN
1      1 2013-01-03   4327       5530.0
2      1 2013-01-04   4486       4327.0
3      1 2013-01-05   4997       4486.0
4      1 2013-01-07   7176       4997.0
5      1 2013-01-08   5580       7176.0
6      1 2013-01-09   5471       5580.0
7      1 2013-01-10   4892       5471.0
8      1 2013-01-11   4881       4892.0
9      1 2013-01-12   4952       4881.0


In [9]:
df["Sales_Lag_7"] = (
    df.groupby("Store")["Sales"].shift(7)
)

In [10]:
df["Sales_Lag_14"] = (
    df.groupby("Store")["Sales"].shift(14)
)

df["Sales_Lag_30"] = (
    df.groupby("Store")["Sales"].shift(30)
)

In [11]:
df["Sales_Rolling_7"] = (
    df.groupby("Store")["Sales"]
      .transform(
          lambda x: x.shift(1).rolling(7).mean()
      )
)

In [12]:
df["Sales_Rolling_14"] = (
    df.groupby("Store")["Sales"]
      .transform(
          lambda x: x.shift(1).rolling(14).mean()
      )
)

df["Sales_Rolling_30"] = (
    df.groupby("Store")["Sales"]
      .transform(
          lambda x: x.shift(1).rolling(30).mean()
      )
)

In [13]:
df["Promo_Active"] = df["Promo"].astype(int)

df["Promo2_Active"] = df["Promo2"].astype(int)

df["Promo_Weekend"] = (
    df["Promo"] * df["IsWeekend"]
)

In [14]:
df["CompetitionDistanceKm"] = (
    df["CompetitionDistance"] / 1000
)

In [15]:
feature_columns = [
    "Date",
    "Store",
    "Sales",
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Sales_Lag_30",
    "Sales_Rolling_7",
    "Sales_Rolling_14",
    "Sales_Rolling_30",
    "Promo",
    "Promo2",
    "IsWeekend",
    "IsHoliday",
    "CompetitionDistanceKm"
]

df[feature_columns].head(40)

,Date,Store,Sales,Sales_Lag_1,Sales_Lag_7,Sales_Lag_14,Sales_Lag_30,Sales_Rolling_7,Sales_Rolling_14,Sales_Rolling_30,Promo,Promo2,IsWeekend,IsHoliday,CompetitionDistanceKm
0,2013-01-02,1,5530,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,1,1.27
1,2013-01-03,1,4327,5530.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,1,1.27
2,2013-01-04,1,4486,4327.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,1,1.27
3,2013-01-05,1,4997,4486.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,1,1.27
4,2013-01-07,1,7176,4997.0,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,1,1.27
5,2013-01-08,1,5580,7176.0,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,1,1.27
6,2013-01-09,1,5471,5580.0,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,1,1.27
7,2013-01-10,1,4892,5471.0,5530.0,NaN,NaN,5366.714286,NaN,NaN,1,0,0,1,1.27
8,2013-01-11,1,4881,4892.0,4327.0,NaN,NaN,5275.571429,NaN,NaN,1,0,0,1,1.27
9,2013-01-12,1,4952,4881.0,4486.0,NaN,NaN,5354.714286,NaN,NaN,0,0,1,0,1.27


In [16]:
lag_columns = [
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Sales_Lag_30",
    "Sales_Rolling_7",
    "Sales_Rolling_14",
    "Sales_Rolling_30"
]

print(df[lag_columns].isnull().sum())

Sales_Lag_1          1115
Sales_Lag_7          7805
Sales_Lag_14        15610
Sales_Lag_30        33450
Sales_Rolling_7      7805
Sales_Rolling_14    15610
Sales_Rolling_30    33450
dtype: int64


In [17]:
df_model = df.dropna(
    subset=[
        "Sales_Lag_30",
        "Sales_Rolling_30"
    ]
).copy()

print("Original rows:", len(df))
print("Model rows:", len(df_model))
print(
    "Removed rows:",
    len(df) - len(df_model)
)

Original rows: 844392
Model rows: 810942
Removed rows: 33450
